Start with [00_end_to_end.ipynb](00_end_to_end.ipynb) for an executable synthetic walkthrough. This topic notebook requires the indicated input checkpoints and dataset-specific parameters. Open from workbench/notebooks/.

# 🧬 01. Single-Cell Data Ingestion & Quality Control

This notebook demonstrates how to load count matrices from the upstream pipeline, calculate quality control metrics, evaluate doublets with Scrublet, and filter low-quality cells.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import sc_workbench as scw
from sc_workbench import SingleCellWorkbench
print(f"sc-workbench version: {scw.__version__}")

### 1. Ingest Count Matrix from Upstream Pipeline

In [ ]:
# Path to count matrix generated by the upstream Smart-seq2 pipeline
matrix_path = "../../data/counts/gene_cell_count_matrix.tsv"

# Initialize workbench
wb = SingleCellWorkbench.from_matrix(matrix_path)
wb

### 2. Calculate Quality Control Metrics

In [ ]:
# Calculates total counts, detected genes, and mito/ribo percentages
wb.calculate_qc(mito_prefix=("MT-", "mt-"))
wb.adata.obs.head()

### 3. Visualize QC Distributions

In [ ]:
scw.plotting.plot_qc_violins(wb.adata)


### 4. Doublet Detection via Scrublet

In [ ]:
# Optional: enable only when appropriate for the library protocol and dataset size.
RUN_SCRUBLET = False
if RUN_SCRUBLET:
    wb.detect_doublets(expected_doublet_rate=0.06)
wb.obs.head(10)

### 5. Filter Low-Quality Cells and Genes

In [ ]:
wb.filter_cells(min_genes=5, min_counts=50, max_pct_mito=25.0)
wb.filter_genes(min_cells=2)
wb

### 6. Save Cleaned AnnData Object

In [ ]:
wb.save("../data/qc_filtered.h5ad")
print("✔ Stage 01 completed successfully!")